In [1]:
# --- Imports ---
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
print("Setup complete.")

Setup complete.


In [2]:
# --- Load Data ---
DATA = '//kaggle/input/competitions/spaceship-titanic'
train = pd.read_csv(f'{DATA}/train.csv')
test  = pd.read_csv(f'{DATA}/test.csv')
print("Train:", train.shape)
print("Test :", test.shape)

Train: (8693, 14)
Test : (4277, 13)


In [3]:
# --- Basic Feature Engineering (no group aggregates) ---
SVC = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def build_basic(df):
    d = df.copy()

    d['GID']  = d['PassengerId'].str.split('_').str[0].astype(int)
    d['Seat'] = d['PassengerId'].str.split('_').str[1].astype(int)

    d['GroupLen'] = d.groupby('GID')['PassengerId'].transform('count')
    d['Solo']     = (d['GroupLen'] == 1).astype(int)

    parts = d['Cabin'].str.split('/', expand=True)
    d['Zone'] = parts[0]
    d['Room'] = pd.to_numeric(parts[1], errors='coerce')
    d['Pier'] = parts[2]

    sp = d[SVC].fillna(0)
    d['Cash']      = sp.sum(axis=1)
    d['CashLog']   = np.log1p(d['Cash'])
    d['ZeroSpend'] = (d['Cash'] == 0).astype(int)
    d['UsedCount'] = (sp > 0).sum(axis=1)

    d['HighEnd']   = d[['Spa', 'VRDeck', 'RoomService']].fillna(0).sum(axis=1)
    d['LowEnd']    = d[['FoodCourt', 'ShoppingMall']].fillna(0).sum(axis=1)
    d['HighRatio'] = d['HighEnd'] / (d['Cash'] + 1)

    d['SleepZero'] = d['CryoSleep'].map({True: 1, False: 0}).fillna(0) * d['ZeroSpend']

    d['AgeBin'] = pd.cut(
        d['Age'], bins=[-1, 12, 18, 30, 50, 200],
        labels=['child', 'teen', 'young', 'mid', 'senior']
    ).astype(str)

    return d

train = build_basic(train)
test  = build_basic(test)
print("Train cols:", train.shape[1])

Train cols: 30


In [4]:
# --- Prepare Features ---
NUMF = [
    'Age', 'GroupLen', 'Solo', 'Seat', 'Room',
    'Cash', 'CashLog', 'ZeroSpend', 'UsedCount',
    'HighEnd', 'LowEnd', 'HighRatio', 'SleepZero'
] + SVC

CATF = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP',
        'Zone', 'Pier', 'AgeBin']

X = train[NUMF + CATF].copy()
y = train['Transported'].astype(int)
X_test = test[NUMF + CATF].copy()

print("X shape     :", X.shape)
print("X_test shape:", X_test.shape)
print("Balance     :", round(y.mean(), 4))

X shape     : (8693, 25)
X_test shape: (4277, 25)
Balance     : 0.5036


In [5]:
# --- Pipeline Builder ---
def make_pipe(model):
    num_step = Pipeline([
        ('fill', SimpleImputer(strategy='mean')),
        ('norm', StandardScaler())
    ])
    cat_step = Pipeline([
        ('fill', SimpleImputer(strategy='most_frequent')),
        ('code', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    prep = ColumnTransformer([
        ('n', num_step, NUMF),
        ('c', cat_step, CATF)
    ])
    return Pipeline([('prep', prep), ('algo', model)])

In [6]:
# --- Single Model Fitter ---
def fit_model(factory, X_tr, y_tr, X_va):
    pipe = make_pipe(factory())
    pipe.fit(X_tr, y_tr)
    return pipe.predict_proba(X_va)[:, 1]

In [7]:
# --- 10-Fold OOF for each model ---
print("=" * 60)
print("10-FOLD OOF PREDICTIONS")
print("=" * 60)

model_factories = [
    ('et',  lambda: ExtraTreesClassifier(
                n_estimators=600, max_depth=14, min_samples_leaf=3,
                random_state=SEED, n_jobs=-1)),
    ('cat', lambda: CatBoostClassifier(
                iterations=800, learning_rate=0.04, depth=6,
                l2_leaf_reg=3.0, bagging_temperature=0.5,
                random_strength=1.0, verbose=0, random_seed=SEED)),
    ('lgbm', lambda: LGBMClassifier(
                n_estimators=800, learning_rate=0.04, num_leaves=40,
                max_depth=8, min_child_samples=25, feature_fraction=0.8,
                subsample=0.85, reg_alpha=0.1, reg_lambda=0.1,
                random_state=SEED, verbose=-1, n_jobs=-1)),
    ('xgb', lambda: XGBClassifier(
                n_estimators=800, learning_rate=0.04, max_depth=6,
                min_child_weight=3, subsample=0.85, colsample_bytree=0.8,
                gamma=0.1, reg_alpha=0.1, reg_lambda=0.1,
                random_state=SEED, n_jobs=-1,
                eval_metric='logloss', tree_method='hist')),
]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
oof = {name: np.zeros(len(X)) for name, _ in model_factories}

for name, factory in model_factories:
    fold_accs = []
    for tr_idx, va_idx in skf.split(X, y):
        probs = fit_model(factory, X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[va_idx])
        oof[name][va_idx] = probs
        fold_accs.append(accuracy_score(y.iloc[va_idx], (probs > 0.5).astype(int)))
    print(f"{name:6} OOF CV: {np.mean(fold_accs):.4f}  (+/- {np.std(fold_accs):.4f})")

10-FOLD OOF PREDICTIONS
et     OOF CV: 0.8056  (+/- 0.0164)
cat    OOF CV: 0.8150  (+/- 0.0163)
lgbm   OOF CV: 0.8077  (+/- 0.0129)
xgb    OOF CV: 0.8088  (+/- 0.0128)


In [8]:
# --- Simple average of top 3 models ---
print("=" * 60)
print("SIMPLE AVERAGE OF TOP 3")
print("=" * 60)

# Rank by OOF
model_oof = {name: accuracy_score(y, (oof[name] > 0.5).astype(int))
             for name, _ in model_factories}

ranked = sorted(model_oof.items(), key=lambda x: x[1], reverse=True)
print("Model OOF accuracies:")
for name, s in ranked:
    print(f"  {name:6}: {s:.4f}")

top3 = [name for name, _ in ranked[:3]]
print(f"\nTop 3: {top3}")

# Simple average
oof_avg = np.mean([oof[name] for name in top3], axis=0)
avg_acc = accuracy_score(y, (oof_avg > 0.5).astype(int))
print(f"\nSimple avg of top 3 OOF: {avg_acc:.4f}")

# Compare against individual models
print("\nAll individual models:")
for name, s in ranked:
    print(f"  {name:6}: {s:.4f}")
print(f"  AVG3  : {avg_acc:.4f}")

SIMPLE AVERAGE OF TOP 3
Model OOF accuracies:
  cat   : 0.8150
  xgb   : 0.8088
  lgbm  : 0.8077
  et    : 0.8056

Top 3: ['cat', 'xgb', 'lgbm']

Simple avg of top 3 OOF: 0.8141

All individual models:
  cat   : 0.8150
  xgb   : 0.8088
  lgbm  : 0.8077
  et    : 0.8056
  AVG3  : 0.8141


In [9]:
# --- Threshold Tuning ---
print("=" * 60)
print("THRESHOLD TUNING")
print("=" * 60)

thresholds = [0.48, 0.49, 0.50, 0.51, 0.52]
best_t, best_acc = 0.5, 0.0

for t in thresholds:
    acc = accuracy_score(y, (oof_avg > t).astype(int))
    print(f"  t = {t:.2f} -> acc = {acc:.4f}")
    if acc > best_acc:
        best_acc, best_t = acc, t

print(f"\nBest threshold: {best_t:.2f}  acc {best_acc:.4f}")

THRESHOLD TUNING
  t = 0.48 -> acc = 0.8123
  t = 0.49 -> acc = 0.8131
  t = 0.50 -> acc = 0.8141
  t = 0.51 -> acc = 0.8131
  t = 0.52 -> acc = 0.8119

Best threshold: 0.50  acc 0.8141


In [10]:
# --- Fit top 3 models on full data with 3-seed averaging ---
print("=" * 60)
print("FITTING TOP 3 MODELS (3 seeds each)")
print("=" * 60)

SEEDS = [42, 2024, 7]
top3_factories = [(n, f) for n, f in model_factories if n in top3]

test_probs = {name: np.zeros(len(X_test)) for name, _ in top3_factories}

for name, factory in top3_factories:
    for seed in SEEDS:
        model = factory()
        # Patch seed
        if hasattr(model, 'random_state'):
            try: model.set_params(random_state=seed)
            except: pass
        if hasattr(model, 'random_seed'):
            try: model.set_params(random_seed=seed)
            except: pass

        pipe = make_pipe(model)
        pipe.fit(X, y)
        test_probs[name] += pipe.predict_proba(X_test)[:, 1] / len(SEEDS)

    print(f"Fitted {name} (3 seeds averaged)")

FITTING TOP 3 MODELS (3 seeds each)
Fitted cat (3 seeds averaged)
Fitted lgbm (3 seeds averaged)
Fitted xgb (3 seeds averaged)


In [11]:
# --- Blend and Predict ---
print("=" * 60)
print("BLEND AND PREDICT")
print("=" * 60)

blended = np.mean([test_probs[name] for name in top3], axis=0)
preds = (blended > best_t)

print("Threshold used:", best_t)
print("Transported    :", int(preds.sum()))
print("Not Transported:", int((~preds).sum()))

BLEND AND PREDICT
Threshold used: 0.5
Transported    : 2123
Not Transported: 2154


In [12]:
# --- Create Submission ---
sub = pd.DataFrame({
    'PassengerId': test['PassengerId'].values,
    'Transported': preds.astype(bool)
})

assert sub.shape == (4277, 2)
assert list(sub.columns) == ['PassengerId', 'Transported']
assert sub['Transported'].dtype == bool
assert sub['PassengerId'].is_unique
assert sub.isna().sum().sum() == 0

sub.to_csv('submission.csv', index=False)
print("submission.csv saved")
print(sub.head())

submission.csv saved
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True


In [13]:
# --- Validate Submission ---
check = pd.read_csv('submission.csv')
print("Exists     :", os.path.exists('submission.csv'))
print("Shape      :", check.shape)
print("Columns    :", list(check.columns))
print("Missing    :", check.isna().sum().sum())
print("Duplicates :", check['PassengerId'].duplicated().sum())
print("Dtype      :", check['Transported'].dtype)
print("\nValue counts:")
print(check['Transported'].value_counts())

Exists     : True
Shape      : (4277, 2)
Columns    : ['PassengerId', 'Transported']
Missing    : 0
Duplicates : 0
Dtype      : bool

Value counts:
Transported
False    2154
True     2123
Name: count, dtype: int64
